# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

In [ ]:
import torch
print("cuda avail:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print("device name:", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"free {free/1e9:.1f} GB / total {total/1e9:.1f} GB")

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "For each [ANS] placeholder in the question, output exactly one separate "
    "\\boxed{} containing only that answer, in order. Do NOT combine multiple "
    "answers into one \\boxed{}. "
    "Provide numeric answers with at least 6 significant figures of precision "
    "(e.g. write 7.79743547584717, not 7.797; write 7091.666667, not 7091.67). "
    "Only put your FINAL answers in \\boxed{} — never use \\boxed{} for "
    "intermediate steps."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and choices, then choose "
    "the single best option. After reasoning, output ONLY the letter inside "
    "\\boxed{}, e.g. \\boxed{C}. Use \\boxed{} only once, at the very end."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    gpu_memory_utilization=0.85,
    max_model_len=12288,
    trust_remote_code=True,
    max_num_seqs=32,
)

sampling_params = SamplingParams(
    n=8,
    max_tokens=8192,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    repetition_penalty=1.0,
)

print("Model loaded.")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

### Generate with Transformers (for Datahub)

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
# def extract_letter(text: str) -> str:
#     m = re.search(r"\\boxed\{([A-Za-z])\}", text)
#     if m:
#         return m.group(1).upper()
#     matches = re.findall(r"\b([A-Z])\b", text.upper())
#     return matches[-1] if matches else ""


# def score_mcq(response: str, gold_letter: str) -> bool:
#     return extract_letter(response) == gold_letter.strip().upper()


# # Load Judger for free-form scoring
# sys.path.insert(0, ".")
# from judger import Judger
# judger = Judger(strict_extract=False)

# results = []
# for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
#     is_mcq = bool(item.get("options"))
#     gold   = item["answer"]

#     if is_mcq:
#         correct = score_mcq(response, str(gold))
#     else:
#         gold_list = gold if isinstance(gold, list) else [gold]
#         try:
#             correct = judger.auto_judge(
#                 pred=response,
#                 gold=gold_list,
#                 options=[[]] * len(gold_list),
#             )
#         except Exception:
#             correct = False

#     results.append({
#         "id":       item.get("id"),
#         "is_mcq":   is_mcq,
#         "gold":     gold,
#         "response": response,
#         "correct":  correct,
#     })

# print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
# mcq_res  = [r for r in results if r["is_mcq"]]
# free_res = [r for r in results if not r["is_mcq"]]

# def acc(subset):
#     return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

# print("=" * 50)
# print("EVALUATION RESULTS")
# print("=" * 50)
# print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
# print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
# print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
# print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
# SAVE_EVAL = True   # Set to False when running on the private test set

# out_path = Path(OUTPUT_PATH)
# out_path.parent.mkdir(parents=True, exist_ok=True)

# with open(out_path, "w") as f:
#     for r in results:
#         if SAVE_EVAL:
#             record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
#                       "response": r["response"], "correct": r["correct"]}
#         else:
#             record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
#         f.write(json.dumps(record) + "\n")

# print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [ ]:
def to_prompt(item):
   system, user = build_prompt(item["question"], item.get("options"))
   return tokenizer.apply_chat_template(
       [{"role": "system", "content": system},
        {"role": "user", "content": user}],
       tokenize=False, add_generation_prompt=True,
   )

test_out = llm.generate([to_prompt(data[0]), to_prompt(data[1])],
                       sampling_params=sampling_params)
print("outputs:", len(test_out), "samples each:", len(test_out[0].outputs))
print("preview:", test_out[0].outputs[0].text[:300])

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm

DATA_PATH = "data/public.jsonl"
RESULTS_PATH = Path("/workspace/results/responses_raw.jsonl")
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
CHUNK = 50

data = [json.loads(line) for line in open(DATA_PATH)]
print(f"Loaded {len(data)} problems")

done_ids = set()
if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
todo = [d for d in data if d["id"] not in done_ids]
print(f"Already done: {len(done_ids)}. Remaining: {len(todo)}.")

def to_prompt(item):
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

with open(RESULTS_PATH, "a") as fout:
    for i in tqdm(range(0, len(todo), CHUNK), desc="Chunks"):
        chunk = todo[i:i+CHUNK]
        chunk_prompts = [to_prompt(item) for item in chunk]
        try:
            outputs = llm.generate(chunk_prompts, sampling_params=sampling_params)
        except Exception as e:
            print(f"Chunk {i//CHUNK} failed: {type(e).__name__}: {e}")
            print("Skipping; will retry on next run.")
            break
        for item, out in zip(chunk, outputs):
            samples = [s.text.strip() for s in out.outputs]
            fout.write(json.dumps({
                "id": item["id"],
                "is_mcq": bool(item.get("options")),
                "samples": samples,
            }) + "\n")
        fout.flush()
        print(f"Chunk done. Total written: {i + len(chunk)}.")

print("All chunks complete.")

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm

PRIVATE_DATA_PATH = "data/private.jsonl"
PRIVATE_RESULTS_PATH = Path("/workspace/results/private_responses.jsonl")
PRIVATE_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
CHUNK = 50

# Load private data
private_data = [json.loads(line) for line in open(PRIVATE_DATA_PATH)]
print(f"Loaded {len(private_data)} private problems")

# Resume support
done_ids = set()
if PRIVATE_RESULTS_PATH.exists():
    with open(PRIVATE_RESULTS_PATH) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
todo = [d for d in private_data if d["id"] not in done_ids]
print(f"Already done: {len(done_ids)}. Remaining: {len(todo)}.")

def to_prompt(item):
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

with open(PRIVATE_RESULTS_PATH, "a") as fout:
    for i in tqdm(range(0, len(todo), CHUNK), desc="Chunks"):
        chunk = todo[i:i+CHUNK]
        chunk_prompts = [to_prompt(item) for item in chunk]
        try:
            outputs = llm.generate(chunk_prompts, sampling_params=sampling_params)
        except Exception as e:
            print(f"Chunk {i//CHUNK} failed: {type(e).__name__}: {e}")
            print("Skipping; will retry on next run.")
            break
        for item, out in zip(chunk, outputs):
            samples = [s.text.strip() for s in out.outputs]
            fout.write(json.dumps({
                "id": item["id"],
                "is_mcq": bool(item.get("options")),
                "samples": samples,
            }) + "\n")
        fout.flush()
        print(f"Chunk done. Total written: {i + len(chunk)}.")

print("All chunks complete.")

In [ ]:
import json, re, csv
from collections import Counter
from pathlib import Path

PRIVATE_DATA_PATH = "data/private.jsonl"
PRIVATE_RESULTS_PATH = Path("/workspace/results/private_responses.jsonl")
SUBMISSION_CSV = Path("/workspace/results/submission.csv")

BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")

def final_answer_key(sample, is_mcq):
    """Hashable representation of a sample's final answer(s) for voting."""
    boxed = BOXED_RE.findall(sample)
    if not boxed:
        return None
    if is_mcq:
        m = re.search(r"([A-Z])", boxed[-1].upper())
        return m.group(1) if m else None
    # Free-form: vote on tuple of last few boxes (we don't know N for private set)
    return tuple(boxed[-3:])

def pick_voted_sample(samples, is_mcq):
    """Pick the sample whose final answer matches the majority vote."""
    keyed = [(final_answer_key(s, is_mcq), s) for s in samples]
    valid = [(k, s) for k, s in keyed if k is not None]
    if not valid:
        return samples[0]  # all 8 samples failed to box anything; submit one anyway
    winner_key = Counter([k for k, _ in valid]).most_common(1)[0][0]
    for k, s in valid:
        if k == winner_key:
            return s
    return samples[0]

# Load private IDs in the original order from private.jsonl
private_ids_in_order = []
with open(PRIVATE_DATA_PATH) as f:
    for line in f:
        private_ids_in_order.append(json.loads(line)["id"])
print(f"Private set has {len(private_ids_in_order)} ids")

# Load all generated responses, indexed by id
records_by_id = {}
with open(PRIVATE_RESULTS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        records_by_id[rec["id"]] = rec
print(f"Generated responses on disk: {len(records_by_id)}")

# Build CSV
missing = []
with open(SUBMISSION_CSV, "w", newline="") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(["id", "response"])
    for rid in private_ids_in_order:
        if rid not in records_by_id:
            missing.append(rid)
            writer.writerow([rid, ""])  # placeholder so the id has a row
            continue
        rec = records_by_id[rid]
        chosen = pick_voted_sample(rec["samples"], rec["is_mcq"])
        writer.writerow([rid, chosen])

print(f"\n✓ Submission written to {SUBMISSION_CSV}")
if missing:
    print(f"⚠️  {len(missing)} ids were missing — submitted as empty:")
    print(f"    First 10: {missing[:10]}")
else:
    print(f"✓ All {len(private_ids_in_order)} ids have responses")

In [ ]:
import json, re
from pathlib import Path

PRIVATE_RESULTS_PATH = Path("/workspace/results/private_responses.jsonl")
BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")

private = [json.loads(l) for l in open(PRIVATE_RESULTS_PATH)]
print(f"Loaded {len(private)} private records")

# A problem is "truncation-failed" if none of its 8 samples produced any boxed answer
truncated_ids = []
for rec in private:
    has_box = any(BOXED_RE.findall(s) for s in rec["samples"])
    if not has_box:
        # All 8 samples failed to produce a boxed answer → likely truncation
        truncated_ids.append(rec["id"])

print(f"Problems with no boxed answer in any sample: {len(truncated_ids)}")
print(f"Examples: {truncated_ids[:10]}")

In [ ]:
import gc, torch
try:
    del llm
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"free {free/1e9:.1f} GB / total {total/1e9:.1f} GB")

In [ ]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    gpu_memory_utilization=0.85,
    max_model_len=24576,            # was 12288, raised to fit 20480-token outputs
    trust_remote_code=True,
    max_num_seqs=16,                # lowered from 32 since each seq is bigger
)

# Note: no enable_lora — we're using baseline model, no SFT adapter
high_tokens_params = SamplingParams(
    n=4,                            # lowered from 8 to keep wall-clock manageable
    max_tokens=20480,                # was 8192 — give hard problems room to finish
    temperature=0.6, top_p=0.95, top_k=20,
    repetition_penalty=1.0,
)
print("vLLM reloaded with high max_tokens.")

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm

PRIVATE_DATA_PATH = "data/private.jsonl"
RERUN_RESULTS_PATH = Path("/workspace/results/private_truncation_rerun.jsonl")
CHUNK = 10  # smaller chunks since each request uses way more GPU

# Load private data and find the truncated subset
private_data = [json.loads(l) for l in open(PRIVATE_DATA_PATH)]
private_by_id = {d["id"]: d for d in private_data}

# Resume support
done_rerun_ids = set()
if RERUN_RESULTS_PATH.exists():
    with open(RERUN_RESULTS_PATH) as f:
        for line in f:
            done_rerun_ids.add(json.loads(line)["id"])

to_rerun = [private_by_id[rid] for rid in truncated_ids if rid not in done_rerun_ids]
print(f"Re-running {len(to_rerun)} truncated problems")

def to_prompt(item):
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

with open(RERUN_RESULTS_PATH, "a") as fout:
    for i in tqdm(range(0, len(to_rerun), CHUNK), desc="Chunks"):
        chunk = to_rerun[i:i+CHUNK]
        chunk_prompts = [to_prompt(item) for item in chunk]
        try:
            outputs = llm.generate(chunk_prompts, sampling_params=high_tokens_params)
        except Exception as e:
            print(f"Chunk failed: {type(e).__name__}: {e}")
            break
        for item, out in zip(chunk, outputs):
            samples = [s.text.strip() for s in out.outputs]
            fout.write(json.dumps({
                "id": item["id"],
                "is_mcq": bool(item.get("options")),
                "samples": samples,
            }) + "\n")
        fout.flush()
print("Truncation rerun complete.")

In [ ]:
import json, csv
from collections import Counter
import re
from pathlib import Path

PRIVATE_DATA_PATH = "data/private.jsonl"
PRIVATE_RESULTS_PATH = Path("/workspace/results/private_responses.jsonl")
RERUN_RESULTS_PATH = Path("/workspace/results/private_truncation_rerun.jsonl")
NEW_SUBMISSION_CSV = Path("/workspace/results/submission_v2.csv")

BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")

def final_answer_key(sample, is_mcq):
    boxed = BOXED_RE.findall(sample)
    if not boxed: return None
    if is_mcq:
        m = re.search(r"([A-Z])", boxed[-1].upper())
        return m.group(1) if m else None
    return tuple(boxed[-3:])

def pick_voted_sample(samples, is_mcq):
    keyed = [(final_answer_key(s, is_mcq), s) for s in samples]
    valid = [(k, s) for k, s in keyed if k is not None]
    if not valid:
        return samples[0]
    winner = Counter([k for k, _ in valid]).most_common(1)[0][0]
    for k, s in valid:
        if k == winner:
            return s
    return samples[0]

# Load all baseline responses
records_by_id = {}
with open(PRIVATE_RESULTS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        records_by_id[rec["id"]] = rec

# Overlay the rerun responses (replacing the original truncated ones)
rerun_count = 0
with open(RERUN_RESULTS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        records_by_id[rec["id"]] = rec   # replace
        rerun_count += 1
print(f"Replaced {rerun_count} truncated records with rerun versions")

# Preserve original private set order
private_ids_in_order = []
with open(PRIVATE_DATA_PATH) as f:
    for line in f:
        private_ids_in_order.append(json.loads(line)["id"])

# Build new CSV
missing = []
with open(NEW_SUBMISSION_CSV, "w", newline="") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(["id", "response"])
    for rid in private_ids_in_order:
        if rid not in records_by_id:
            missing.append(rid)
            writer.writerow([rid, ""])
            continue
        rec = records_by_id[rid]
        chosen = pick_voted_sample(rec["samples"], rec["is_mcq"])
        writer.writerow([rid, chosen])

print(f"Written to {NEW_SUBMISSION_CSV}")
if missing:
    print(f"WARNING: {len(missing)} ids missing — submitted as empty")